In [1]:
from dataset import load_dataset
import numpy as np
from tqdm import tqdm
import pandas as pd
from regression.probabilistic_rf_scoring import fit_rank_pdfs_loglik
from score_PHM import get_regression_score
import joblib
import os
from regression.RandomForest.random_forest_regression import RandomForestRegressorModel
from regression.PolynomialRegressor.polynomial_regression import PolynomialRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from regression.GPR.gpr_model import GPRModel

In [2]:
TARGET_COL = "trq_target"
FEATURES=["trq_measured","mgt","ias","oat","np_ng_ratio","pa"]
TRAIN_PATH_SPLIT= "../../data/processed/split_85_15/train_85.csv"
TEST_PATH_SPLIT= "../../data/processed/split_85_15/test_15.csv"
OUTPUT_PATH_RANDOM_FOREST= "../../output/random_forest_regressor/model_target/random_forest_model_target.pkl"
OUTPUT_PATH_POLINOMIAL= "../../output/polynomial_regression/model_target/polynomial_regression_model_target.pkl"
OUTPUT_PATH_GPR = "../../output/GPR/margin/gpr_model_5000.pkl"
OUTPUT_PATH_GPR_TARGET = "../../output/GPR/model_target/gpr_model_5000.pkl"
OUTPUT_PATH_RESIDUI="../../output/polynomial_regression/model_target/residui.csv"
OUTPUT_PATH_PR_TEST_10="../../output/polynomial_regression/model_target/test10.csv"
OUTPUT_PATH_RM_TEST_10="../../output/random_forest_regressor/model_target/test10.csv"
OUTPUT_PATH_GPR_TEST_10="../../output/GPR/margin/test10.csv"
OUTPUT_PATH_GPR_TARGET_TEST_10="../../output/GPR/model_target/test10.csv"
ITERATIONS=1000
PDF_LIST = ["norm", "t", "cauchy", "laplace", "lognorm", "gamma"]


In [3]:
train_df = load_dataset(TRAIN_PATH_SPLIT)
test_df = load_dataset(TEST_PATH_SPLIT)
X_train_90, X_test_10 = train_test_split(
    test_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)
X_test_10.head()

,trq_measured,oat,mgt,pa,ias,trq_margin,trq_target,np_ng_ratio,faulty
49287,90.4,-17.75,640.1,442.2648,126.2500,-18.310882,110.663455,0.947185,1
82653,69.6,21.50,620.4,568.4520,111.0625,3.316883,67.365563,1.052271,0
46485,72.6,6.25,567.5,-54.2544,96.5000,9.158095,66.509039,0.922861,0
76933,58.2,20.25,572.1,17.9832,82.3750,6.743027,54.523468,1.097561,0
35083,61.6,19.00,595.3,146.6088,67.6250,-1.956771,62.829428,1.078990,1


In [ ]:
rf_model = RandomForestRegressorModel(
    n_estimators=500,
    min_samples_leaf=5
)
if os.path.exists(OUTPUT_PATH_RANDOM_FOREST):
    rf_model=joblib.load(OUTPUT_PATH_RANDOM_FOREST)
    print("Caricato")

In [ ]:
poly_model = PolynomialRegression(degree=2)
if os.path.exists(OUTPUT_PATH_POLINOMIAL):
    poly_model= joblib.load(OUTPUT_PATH_POLINOMIAL)
    print("Caricato")

In [11]:
if not os.path.exists(OUTPUT_PATH_PR_TEST_10):
    residui=load_dataset(OUTPUT_PATH_RESIDUI)
    residual_arr = residui["residual"].astype(float).to_numpy()
    rng = np.random.default_rng(42)

    rows_out = []

    pbar = tqdm(range(len(X_test_10)), desc="Processing test set", unit="row")

    for i in pbar:

        test_row = X_test_10.iloc[i]
        trq_measured = float(test_row["trq_measured"])
        trq_target_true = float(test_row["trq_target"])
        trq_margin_true = float(test_row["trq_margin"])

        test_x = test_row[FEATURES].to_frame().T

        pbar.set_postfix_str("predicting")
        trq_target_pred = float(poly_model.predict(test_x)[0])
        trq_margin_pred = (trq_measured / trq_target_pred - 1.0) * 100.0

        pbar.set_postfix_str("sampling residuals")
        sampled_res = rng.choice(residual_arr, size=int(ITERATIONS), replace=True)
        trq_target_prob = trq_target_pred + sampled_res

        trq_target_prob = np.where(np.abs(trq_target_prob) < 1e-12, np.nan, trq_target_prob)

        trq_margin_dist = (trq_measured / trq_target_prob - 1.0) * 100.0
        trq_margin_dist = trq_margin_dist[np.isfinite(trq_margin_dist)]


        pbar.set_postfix_str("fitting pdf")
        ranking, best = fit_rank_pdfs_loglik(trq_margin_dist, pdf_list=PDF_LIST)

        if best is None or best.get("pdf_args") is None:
            continue

        best_pdf = best["pdf_type"]
        best_args = best["pdf_args"]

        pbar.set_postfix_str("scoring")
        score_true = float(get_regression_score(best_pdf, best_args, float(trq_margin_true)))
        score_pred = float(get_regression_score(best_pdf, best_args, float(trq_margin_pred)))

        out = test_row[FEATURES].to_dict()
        out.update({
            "idx": i,
            "trq_target_true": trq_target_true,
            "trq_target_pred": trq_target_pred,
            "trq_margin_true": float(trq_margin_true),
            "trq_margin_pred": float(trq_margin_pred),
            "best_pdf": best_pdf,
            "best_pdf_args": best_args,
            "best_loglik": float(best.get("loglik", np.nan)),
            "score_true": score_true,
            "score_pred": score_pred,
            "n_mc_used": int(trq_margin_dist.size),
        })
        rows_out.append(out)
    pbar.close()
    PL_test_10 = pd.DataFrame(rows_out)
    PL_test_10.to_csv(OUTPUT_PATH_PR_TEST_10)
else:
    print("Test gia presente")
    PL_test_10=load_dataset(OUTPUT_PATH_PR_TEST_10)

Test gia presente


In [12]:
score_mean=np.mean(PL_test_10["score_true"])
print(score_mean)

0.5341282159161977


In [13]:
if not os.path.exists(OUTPUT_PATH_RM_TEST_10):
    rows_out = []
    pbar = tqdm(range(len(X_test_10)), desc="Processing test set", unit="row")

    for i in pbar:
        test_row = X_test_10.iloc[i]

        trq_measured = float(test_row["trq_measured"])
        trq_target_true = float(test_row["trq_target"])
        trq_margin_true = float(test_row["trq_margin"])

        # 1) features del campione (1, n_features)
        x0 = test_row[FEATURES].to_numpy().reshape(1, -1)

        # 2) predizioni dei singoli alberi come DF (n_trees righe, 1 colonna)
        pbar.set_postfix_str("predict trees")
        tree_preds = rf_model.predict_trees(x0)  # -> DataFrame con 'trq_target_prediction'

        # 3) torque margin per ogni albero (distribuzione)
        tree_preds["trq_margin_pred"] = (trq_measured / tree_preds["trq_target_prediction"] - 1.0) * 100.0

        # distribuzione (array) e media
        trq_margin_dist = tree_preds["trq_margin_pred"].to_numpy(dtype=np.float64)
        trq_margin_pred = float(trq_margin_dist.mean())

        # target pred medio (media alberi)
        trq_target_pred = float(tree_preds["trq_target_prediction"].to_numpy(dtype=np.float64).mean())

        # 4) fit pdf sulla distribuzione (NON sulla media)
        pbar.set_postfix_str("fitting pdf")
        ranking, best = fit_rank_pdfs_loglik(trq_margin_dist, pdf_list=PDF_LIST)

        if best is None or best.get("pdf_args") is None:
            continue

        best_pdf = best["pdf_type"]
        best_args = best["pdf_args"]

        # 5) scoring
        pbar.set_postfix_str("scoring")
        score_true = float(get_regression_score(best_pdf, best_args, trq_margin_true))
        score_pred = float(get_regression_score(best_pdf, best_args, trq_margin_pred))

        # 6) output row
        out = test_row[FEATURES].to_dict()
        out.update({
            "idx": i,
            "trq_target_true": trq_target_true,
            "trq_target_pred": trq_target_pred,
            "trq_margin_true": trq_margin_true,
            "trq_margin_pred": trq_margin_pred,
            "best_pdf": best_pdf,
            "best_pdf_args": best_args,
            "best_loglik": float(best.get("loglik", np.nan)),
            "score_true": score_true,
            "score_pred": score_pred,
            "n_mc_used": int(trq_margin_dist.size),
        })

        rows_out.append(out)

    pbar.close()

    RM_test_10 = pd.DataFrame(rows_out)
    RM_test_10.head()
    RM_test_10.to_csv(OUTPUT_PATH_RM_TEST_10)
else:
    RM_test_10 = load_dataset(OUTPUT_PATH_RM_TEST_10)


In [14]:
score_mean_RM=np.mean(RM_test_10["score_true"])
print(score_mean_RM)

0.008118338135839587


GPR TARGET

In [7]:
scaler = StandardScaler()
test_X=scaler.fit_transform(test_df[FEATURES])
test_y_target=test_df[TARGET_COL]

In [8]:
gpr_model_target = GPRModel(random_state=42)
if os.path.exists(OUTPUT_PATH_GPR_TARGET):
    print("Modello pre-addestrato trovato! Caricamento in corso...")
    try:
        gpr_model_target = joblib.load(OUTPUT_PATH_GPR_TARGET)
        print("SUCCESSO: Modello caricato correttamente.")
        addestramento_necessario = False
    except Exception as e:
        print(f"Errore nel caricamento dei file (file corrotti?): {e}")
        addestramento_necessario = True
else:
    print("Modello non trovato o incompleto. Avvio procedura di addestramento...")
    addestramento_necessario = True

Modello pre-addestrato trovato! Caricamento in corso...
SUCCESSO: Modello caricato correttamente.


In [9]:
if not os.path.exists(OUTPUT_PATH_GPR_TARGET_TEST_10):
    rows_out = []
    pbar = tqdm(range(len(test_X)), desc="Processing test set", unit="row")

    for i in pbar :
        test_row = test_X[i:i+1]
        trq_measured_real = float(test_df.iloc[i]['trq_measured'])
        trq_margin_true = float(test_df.iloc[i]['trq_margin'])
        mu,std = gpr_model_target.predict(test_row)
        mu_target=float(mu[0])
        std_target=float(std[0])
        mu_margin = 100 * (trq_measured_real / mu_target - 1)
        # 3. propagazione dell'incertezza con il Metodo Delta
        std_margin = np.abs(-100 * trq_measured_real / (mu_target**2)) * std_target
        pdf_args = {"loc": mu_margin, "scale": std_margin}

        pbar.set_postfix_str("scoring")
        score_true = float(get_regression_score("norm", pdf_args, trq_margin_true))

        score_pred = float(get_regression_score("norm", pdf_args, mu_margin))

        out={
            "idx": i,
            "trq_margin_true": trq_margin_true,
            "trq_margin_pred": mu_margin,
            "pdf": "norm",
            "pdf_args": pdf_args,
            "score_true": score_true,
            "score_pred": score_pred,
        }
        rows_out.append(out)

    pbar.close()

    GPR_target_test_10 = pd.DataFrame(rows_out)
    GPR_target_test_10.to_csv(OUTPUT_PATH_GPR_TARGET_TEST_10)
else:
    GPR_target_test_10 = load_dataset(OUTPUT_PATH_GPR_TARGET_TEST_10)

Processing test set: 100%|██████████| 111394/111394 [16:50<00:00, 110.27row/s, scoring]


In [10]:
score_mean_GPR_target=np.mean(GPR_target_test_10["score_true"])
print(score_mean_GPR_target)

0.657137793979394


GPR Margin

In [13]:
test_X=scaler.fit_transform(test_df[FEATURES])
test_y=test_df["trq_margin"]

In [ ]:
gpr_model = GPRModel(random_state=42)
if os.path.exists(OUTPUT_PATH_GPR):
    print("Modello pre-addestrato trovato! Caricamento in corso...")
    try:
        gpr_model = joblib.load(OUTPUT_PATH_GPR)
        print("SUCCESSO: Modello caricato correttamente.")
        addestramento_necessario = False
    except Exception as e:
        print(f"Errore nel caricamento dei file (file corrotti?): {e}")
        addestramento_necessario = True
else:
    print("Modello non trovato o incompleto. Avvio procedura di addestramento...")
    addestramento_necessario = True

In [ ]:
if not os.path.exists(OUTPUT_PATH_GPR_TEST_10):
    rows_out = []
    pbar = tqdm(range(len(test_X)), desc="Processing test set", unit="row")

    for i in pbar:
        test_row = test_X[i:i+1]
        trq_margin_true = float(test_y[i])
        pbar.set_postfix_str("predict trees")
        mu,std = gpr_model.predict(test_row)
        mu_margin=float(mu[0])
        std_margin=float(std[0])
        pdf_args = {"loc": mu_margin, "scale": std_margin}

        pbar.set_postfix_str("scoring")
        score_true = float(get_regression_score("norm", pdf_args, trq_margin_true))

        score_pred = float(get_regression_score("norm", pdf_args, mu_margin))

        out={
            "idx": i,
            "trq_margin_true": trq_margin_true,
            "trq_margin_pred": mu_margin,
            "pdf": "norm",
            "pdf_args": pdf_args,
            "score_true": score_true,
            "score_pred": score_pred,
        }
        rows_out.append(out)

    pbar.close()

    GPR_test_10 = pd.DataFrame(rows_out)
    GPR_test_10.to_csv(OUTPUT_PATH_GPR_TEST_10)
else:
    GPR_test_10 = load_dataset(OUTPUT_PATH_GPR_TEST_10)

In [ ]:
GPR_test_10.to_csv(OUTPUT_PATH_GPR_TEST_10)

In [ ]:
score_mean_GPR=np.mean(GPR_test_10["score_true"])
print(score_mean_GPR)